# NER-Goldstandard-Evaluation (FIGUR) mit Stoppregel


## Konfiguration


In [ ]:
from pathlib import Path

GOLD_PFAD   = Path("..") / "04_konsolidiert" / "goldstandard_v1_ner_final.conll"
META_PFAD   = Path("..") / "02_quelldaten" / "goldstandard_v1_meta.csv"
MODELL      = Path("..") / ".." / "output" / "model-best"
TRAIN_SPACY = Path("..") / ".." / "corpus" / "train.spacy"   
FORMEN_TXT  = None                          

LABEL = "FIGUR"

BASELINE = "de_core_news_lg"
BASELINE_ABBILDUNG = {"PER": LABEL}

REPS, SEED = 1000, 42
AUSGABE = Path("..") / "05_ergebnisse" / "ner"
AUSGABE.mkdir(parents=True, exist_ok=True)

ZIEL_HALBBREITE_GESAMT   = 5.0
ZIEL_HALBBREITE_SAMMLUNG = 8.0

IAA_F1 = {"Tranche 1": 0.877, "Tranche 2": 0.924}
IAA_GEWICHTET = 0.904        
IAA_HALBBREITE = 3.8         

MIN_CLUSTER = 8

print(f"Gold: {GOLD_PFAD} | Modell: {MODELL} | Baseline: {BASELINE}")
print(f"Ziele: ±{ZIEL_HALBBREITE_GESAMT} gesamt / "
      f"±{ZIEL_HALBBREITE_SAMMLUNG} je Sammlung")
print(f"Agreement-Deckel: {IAA_GEWICHTET:.3f} (gewichtet), "
      f"je Tranche {IAA_F1}")
print()
print("Vorabfestlegung: Das Ergebnis dieses Laufs wird berichtet, wie es")
print("ausfaellt, einschliesslich des Vergleichs mit der Baseline.")

Gold: ../04_konsolidiert/goldstandard_v1_ner_final.conll | Modell: ../../output/model-best | Baseline: de_core_news_lg
Ziele: ±5.0 gesamt / ±8.0 je Sammlung
Agreement-Deckel: 0.904 (gewichtet), je Tranche {'Tranche 1': 0.877, 'Tranche 2': 0.924}

Vorabfestlegung: Das Ergebnis dieses Laufs wird berichtet, wie es
ausfaellt, einschliesslich des Vergleichs mit der Baseline.


## Schritt 1 — Gold-CoNLL und Meta einlesen



In [2]:
import csv, sys

def read_conll(path):
    sents, tokens, tags, erz = [], [], [], None
    with open(path, encoding="utf-8") as f:
        for lineno, raw in enumerate(f, 1):
            line = raw.rstrip("\n")
            if line.startswith("#"):
                if "=" in line:
                    key, _, val = line.lstrip("#").partition("=")
                    if key.strip().lower() in ("erzaehlung", "erzählung"):
                        erz = val.strip()
                continue
            if not line.strip():
                if tokens:
                    sents.append(dict(erzaehlung=erz or "UNBEKANNT",
                                      tokens=tokens, tags=tags))
                    tokens, tags = [], []
                continue
            parts = line.split("\t")
            if len(parts) < 3:
                raise ValueError(f"{path}:{lineno}: 'Token\\tPOS\\tNER' "
                                 f"erwartet: {line!r}")
            tokens.append(parts[0]); tags.append(parts[2])
    if tokens:
        sents.append(dict(erzaehlung=erz or "UNBEKANNT",
                          tokens=tokens, tags=tags))
    return sents

def read_meta(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return {r["erzaehlung"].strip(): r["sammlung"].strip()
                for r in csv.DictReader(f)}

sents = read_conll(GOLD_PFAD)
meta = read_meta(META_PFAD)
fehlend = {s["erzaehlung"] for s in sents} - set(meta)
if fehlend:
    print(f"WARNUNG: ohne Sammlung in Meta: {sorted(fehlend)}")
n_gold = sum(1 for s in sents for t in s["tags"] if t.startswith("B-"))
print(f"{len(sents)} Sätze | {len({s['erzaehlung'] for s in sents})} "
      f"Erzählungen | {n_gold} Gold-Entitäten")

150 Sätze | 20 Erzählungen | 202 Gold-Entitäten


## Schritt 2 — BIO → Spannen


In [3]:
def bio_to_spans(tags):
    spans, start = set(), None
    for i, t in enumerate(tags):
        if t.startswith("B-"):
            if start is not None: spans.add((start, i))
            start = i
        elif t.startswith("I-"):
            if start is None: start = i
        else:
            if start is not None: spans.add((start, i)); start = None
    if start is not None: spans.add((start, len(tags)))
    return spans

print("Beispiel:", bio_to_spans(["O","B-FIGUR","I-FIGUR","O","B-FIGUR"]))

Beispiel: {(4, 5), (1, 3)}


## Schritt 3 — Trainingsformen für die Seen/Unseen-Diagnostik



In [4]:
def normalize(form):
    return " ".join(form.split()).lower()

train_set = None
if TRAIN_SPACY or FORMEN_TXT:
    train_set = set()
    if TRAIN_SPACY:
        import spacy
        from spacy.tokens import DocBin
        _nlp_tmp = spacy.blank("de")
        for doc in DocBin().from_disk(TRAIN_SPACY).get_docs(_nlp_tmp.vocab):
            for e in doc.ents:
                train_set.add(normalize(e.text))
    if FORMEN_TXT:
        train_set |= {normalize(l) for l in open(FORMEN_TXT, encoding="utf-8")
                      if l.strip()}
    print(f"Trainingsformen: {len(train_set)}")
else:
    print("Seen/Unseen-Diagnostik übersprungen (keine Quelle konfiguriert).")

Trainingsformen: 450


## Schritt 4 — Modellvorhersage auf der Gold-Tokenisierung



In [ ]:
import spacy
from spacy.tokens import Doc

nlp = spacy.load(MODELL)
docs = [Doc(nlp.vocab, words=s["tokens"]) for s in sents]
preds = []
for doc in nlp.pipe(docs, batch_size=64):
    preds.append({(e.start, e.end) for e in doc.ents if e.label_ == LABEL})
print(f"Vorhersagen: {sum(len(p) for p in preds)} {LABEL}-Spannen")

preds_basis = None
if BASELINE:
    nlp_b = spacy.load(BASELINE)
    docs_b = [Doc(nlp_b.vocab, words=s["tokens"]) for s in sents]
    preds_basis = []
    for doc in nlp_b.pipe(docs_b, batch_size=64):
        preds_basis.append({(e.start, e.end) for e in doc.ents
                            if BASELINE_ABBILDUNG.get(e.label_) == LABEL})
    print(f"Baseline {BASELINE}: "
          f"{sum(len(p) for p in preds_basis)} Spannen nach Abbildung "
          f"{BASELINE_ABBILDUNG}")

Vorhersagen: 201 FIGUR-Spannen
Baseline de_core_news_lg: 33 Spannen nach Abbildung {'PER': 'FIGUR'}


## Schritt 5 — Zähler, Seen/Unseen und Fehlerliste


In [6]:
from collections import defaultdict
import numpy as np

counts = defaultdict(lambda: np.zeros(3, dtype=np.int64))  # [tp, fp, fn]
counts_basis = defaultdict(lambda: np.zeros(3, dtype=np.int64))
seen, new = dict(tp=0, fn=0), dict(tp=0, fn=0)
fp_seen = fp_new = 0
fehler = []

basis_iter = preds_basis if preds_basis is not None else [None] * len(sents)
for s, pspans, bspans in zip(sents, preds, basis_iter):
    gspans = bio_to_spans(s["tags"])
    c = counts[s["erzaehlung"]]
    c[0] += len(gspans & pspans)
    c[1] += len(pspans - gspans)
    c[2] += len(gspans - pspans)
    if bspans is not None:
        cb = counts_basis[s["erzaehlung"]]
        cb[0] += len(gspans & bspans)
        cb[1] += len(bspans - gspans)
        cb[2] += len(gspans - bspans)
    form = lambda sp: " ".join(s["tokens"][sp[0]:sp[1]])
    if train_set is not None:
        for sp in gspans:
            b = seen if normalize(form(sp)) in train_set else new
            b["tp" if sp in pspans else "fn"] += 1
        for sp in pspans - gspans:
            if normalize(form(sp)) in train_set: fp_seen += 1
            else: fp_new += 1
    for sp in sorted(pspans - gspans):
        fehler.append(dict(typ="FP", erzaehlung=s["erzaehlung"],
                           form=form(sp), satz=" ".join(s["tokens"])))
    for sp in sorted(gspans - pspans):
        fehler.append(dict(typ="FN", erzaehlung=s["erzaehlung"],
                           form=form(sp), satz=" ".join(s["tokens"])))

print(f"Erzählungen mit Zählern: {len(counts)} | "
      f"Fehlerlisteneinträge: {len(fehler)}")

Erzählungen mit Zählern: 20 | Fehlerlisteneinträge: 77


## Schritt 6 — Aggregation mit Cluster-Bootstrap


In [ ]:
rng = np.random.default_rng(SEED)

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

def cluster_ci(arr, idx=None):
    leer = {k: (float("nan"), float("nan")) for k in ("precision", "recall", "f1")}
    if len(arr) < 2:
        return leer
    if idx is None:
        idx = rng.integers(0, len(arr), size=(REPS, len(arr)))
    s = arr[idx].sum(axis=1)
    tp, fp, fn = s[:, 0], s[:, 1], s[:, 2]
    werte = {
        "precision": np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0),
        "recall":    np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0),
        "f1":        np.where(2 * tp + fp + fn > 0,
                              2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0),
    }
    return {k: tuple(float(x) for x in np.percentile(v, [2.5, 97.5]))
            for k, v in werte.items()}

erz_ids = sorted(counts)
def report(name, ids):
    arr = np.stack([counts[e] for e in ids])
    tp, fp, fn = arr.sum(0)
    p, r, f = prf(tp, fp, fn)
    ci = cluster_ci(arr)
    return dict(ebene=name, k=len(ids), gold=int(tp + fn), tp=int(tp),
                fp=int(fp), fn=int(fn), precision=p, recall=r, f1=f,
                ci=ci["f1"], ci_precision=ci["precision"],
                ci_recall=ci["recall"])

rows = [report("GESAMT", erz_ids)]
sammlungen = defaultdict(list)
for e in erz_ids:
    sammlungen[meta.get(e, "UNBEKANNT")].append(e)
for name in sorted(sammlungen):
    rows.append(report(f"Sammlung: {name}", sammlungen[name]))

def spanne(paar):
    return ("—" if paar[0] != paar[0]
            else f"[{paar[0]:.3f}, {paar[1]:.3f}]")

print(f"{'Ebene':<24}{'k':>3}{'Gold':>6}{'P':>7}{'P 95%-CI':>17}"
      f"{'R':>7}{'R 95%-CI':>17}{'F1':>7}{'F1 95%-CI':>17}")
for r in rows:
    print(f"{r['ebene']:<24}{r['k']:>3}{r['gold']:>6}"
          f"{r['precision']:>7.3f}{spanne(r['ci_precision']):>17}"
          f"{r['recall']:>7.3f}{spanne(r['ci_recall']):>17}"
          f"{r['f1']:>7.3f}{spanne(r['ci']):>17}")

#Warnung bei zu wenigen Clustern: das Perzentilintervall faellt dann
#zu eng aus. Simulierte Ueberdeckung bei fuenf Erzaehlungen rund 0,83,
#bei acht rund 0,88, bei zwanzig rund 0,91 statt der nominellen 0,95.
schwach = [r["ebene"] for r in rows if 2 <= r["k"] < MIN_CLUSTER]
if schwach:
    print()
    print(f"HINWEIS: Weniger als {MIN_CLUSTER} Erzaehlungen bei: "
          + ", ".join(schwach) + ".")
    print("Die Intervalle dieser Ebenen sind zu optimistisch und nur als")
    print("Richtungsangabe zu lesen, nicht als belastbare Schaetzung.")

print("\nPro Erzählung (Punktschätzer):")
for e in erz_ids:
    tp, fp, fn = counts[e]
    p, r, f = prf(tp, fp, fn)
    print(f"  {e:<40}{meta.get(e,'?'):<11}Gold={int(tp+fn):>3} F1={f:.2f}")

Ebene                     k  Gold      P         P 95%-CI      R         R 95%-CI     F1        F1 95%-CI
GESAMT                   20   202  0.811   [0.742, 0.877]  0.807   [0.721, 0.880]  0.809   [0.745, 0.866]
Sammlung: Andersen        5    43  0.744   [0.614, 0.878]  0.744   [0.622, 0.883]  0.744   [0.644, 0.854]
Sammlung: Arabisch        5    57  0.821   [0.727, 0.939]  0.807   [0.738, 0.881]  0.814   [0.745, 0.885]
Sammlung: Grimm           5    50  0.904   [0.825, 0.956]  0.940   [0.897, 0.981]  0.922   [0.860, 0.966]
Sammlung: Japanisch       5    52  0.760   [0.608, 0.938]  0.731   [0.518, 0.920]  0.745   [0.593, 0.868]

HINWEIS: Weniger als 8 Erzaehlungen bei: Sammlung: Andersen, Sammlung: Arabisch, Sammlung: Grimm, Sammlung: Japanisch.
Die Intervalle dieser Ebenen sind zu optimistisch und nur als
Richtungsangabe zu lesen, nicht als belastbare Schaetzung.

Pro Erzählung (Punktschätzer):
  0_Alaeddin und die Wunderlampe          Arabisch   Gold=  9 F1=0.88
  0_Die Abenteuer Sin

## Schritt 5b — Vergleich mit der Baseline (gepaart)


In [ ]:
if preds_basis is None:
    print("Kein Baseline-Vergleich konfiguriert (BASELINE = None).")
else:
    arr_m = np.stack([counts[e] for e in erz_ids])
    arr_b = np.stack([counts_basis[e] for e in erz_ids])

    tpb, fpb, fnb = arr_b.sum(0)
    pb, rb, fb = prf(tpb, fpb, fnb)
    fm = rows[0]["f1"]

    #Dieselbe Ziehung für beide Systeme: das ist der gepaarte Teil
    idx = rng.integers(0, len(arr_m), size=(REPS, len(arr_m)))
    ci_m = cluster_ci(arr_m, idx)["f1"]
    ci_b = cluster_ci(arr_b, idx)["f1"]

    def f1_reihe(arr):
        s = arr[idx].sum(axis=1)
        tp, fp, fn = s[:, 0], s[:, 1], s[:, 2]
        return np.where(2 * tp + fp + fn > 0,
                        2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)

    diff = f1_reihe(arr_m) - f1_reihe(arr_b)
    d_lo, d_hi = np.percentile(diff, [2.5, 97.5])

    print(f"{'System':<34}{'P':>7}{'R':>7}{'F1':>7}{'95%-CI (F1)':>20}")
    print(f"{'feingetunt (' + str(MODELL.name) + ')':<34}"
          f"{rows[0]['precision']:>7.3f}{rows[0]['recall']:>7.3f}{fm:>7.3f}"
          f"{'[' + f'{ci_m[0]:.3f}, {ci_m[1]:.3f}' + ']':>20}")
    print(f"{'Baseline (' + BASELINE + ', PER->' + LABEL + ')':<34}"
          f"{pb:>7.3f}{rb:>7.3f}{fb:>7.3f}"
          f"{'[' + f'{ci_b[0]:.3f}, {ci_b[1]:.3f}' + ']':>20}")
    print()
    print(f"Gepaarte Differenz F1: {fm - fb:+.3f}  "
          f"95%-CI [{d_lo:+.3f}, {d_hi:+.3f}]")
    if d_lo > 0:
        print("Die Null liegt ausserhalb: der Vorsprung ist statistisch "
              "abgesichert.")
    elif d_hi < 0:
        print("Die Null liegt ausserhalb, das Vorzeichen ist jedoch "
              "negativ: die Baseline ist besser.")
    else:
        print("Die Null liegt im Intervall: der Unterschied ist nicht "
              "abgesichert.")
    print()
    print(f"Zum Vergleich die ungepaarte Betrachtung: Halbbreiten "
          f"±{100*(ci_m[1]-ci_m[0])/2:.1f} und "
          f"±{100*(ci_b[1]-ci_b[0])/2:.1f} Punkte gegenueber "
          f"±{100*(d_hi-d_lo)/2:.1f} Punkten fuer die Differenz.")

System                                  P      R     F1         95%-CI (F1)
feingetunt (model-best)             0.811  0.807  0.809      [0.746, 0.867]
Baseline (de_core_news_lg, PER->FIGUR)  0.576  0.094  0.162      [0.080, 0.238]

Gepaarte Differenz F1: +0.647  95%-CI [+0.524, +0.771]
Die Null liegt ausserhalb: der Vorsprung ist statistisch abgesichert.

Zum Vergleich die ungepaarte Betrachtung: Halbbreiten ±6.0 und ±7.9 Punkte gegenueber ±12.3 Punkten fuer die Differenz.


## Schritt 7 — Stoppregel prüfen


In [ ]:
print("Stoppregel-Prüfung (Ziel: "
      f"±{ZIEL_HALBBREITE_GESAMT} gesamt / ±{ZIEL_HALBBREITE_SAMMLUNG} "
      "je Sammlung, F1-Punkte):\n")
alles_ok = True
for r in rows:
    lo, hi = r["ci"]
    if lo != lo:
        print(f"  {r['ebene']:<26} —  kein CI (k<2)")
        alles_ok = False
        continue
    hb = 100 * (hi - lo) / 2
    ziel = (ZIEL_HALBBREITE_GESAMT if r["ebene"] == "GESAMT"
            else ZIEL_HALBBREITE_SAMMLUNG)
    ok = hb <= ziel
    alles_ok &= ok
    print(f"  {r['ebene']:<26} ±{hb:4.1f}  Ziel ±{ziel:.0f}  "
          f"{'ERREICHT' if ok else 'nicht erreicht'}")
print("\n=> " + ("Zielpräzision erreicht — Annotation kann stoppen."
                 if alles_ok else
                 "Zielpräzision nicht erreicht; ohne weitere Tranche ist das "
                 "offenzulegen."))

# Agreement-Deckel: zwei mechanische Prüfungen
print()
print("Prüfung des Agreement-Deckels:")
f1_gesamt = rows[0]["f1"]
if f1_gesamt > IAA_GEWICHTET:
    print(f"  WARNUNG: Modell-F1 {f1_gesamt:.3f} liegt ueber dem Agreement "
          f"{IAA_GEWICHTET:.3f}.")
    print("  Ein Wert oberhalb des menschlichen Agreements belegt keine")
    print("  Ueberlegenheit, sondern deutet auf einen unzuverlaessigen")
    print("  Massstab oder auf Trainingsmaterial im Goldstandard hin.")
else:
    print(f"  Modell-F1 {f1_gesamt:.3f} liegt unter dem Agreement "
          f"{IAA_GEWICHTET:.3f} — unauffaellig.")

hb_gesamt = 100 * (rows[0]["ci"][1] - rows[0]["ci"][0]) / 2
if hb_gesamt == hb_gesamt and hb_gesamt < IAA_HALBBREITE:
    print(f"  WARNUNG: Das Intervall (±{hb_gesamt:.1f}) ist enger als die "
          f"Annotationsunsicherheit (±{IAA_HALBBREITE:.1f}).")
    print("  Eine solche Praezision ist nicht belegbar, weil schon der")
    print("  Massstab selbst unschaerfer ist (Scheinpraezision).")
else:
    print(f"  Intervall ±{hb_gesamt:.1f} gegenueber Annotationsunsicherheit "
          f"±{IAA_HALBBREITE:.1f} — unauffaellig.")

Stoppregel-Prüfung (Ziel: ±5.0 gesamt / ±8.0 je Sammlung, F1-Punkte):

  GESAMT                     ± 6.0  Ziel ±5  nicht erreicht
  Sammlung: Andersen         ±10.5  Ziel ±8  nicht erreicht
  Sammlung: Arabisch         ± 7.0  Ziel ±8  ERREICHT
  Sammlung: Grimm            ± 5.3  Ziel ±8  ERREICHT
  Sammlung: Japanisch        ±13.8  Ziel ±8  nicht erreicht

=> Zielpräzision nicht erreicht; ohne weitere Tranche ist das offenzulegen.

Prüfung des Agreement-Deckels:
  Modell-F1 0.809 liegt unter dem Agreement 0.904 — unauffaellig.
  Intervall ±6.0 gegenueber Annotationsunsicherheit ±3.8 — unauffaellig.


## Schritt 8 — Seen/Unseen-Diagnostik und Export


In [10]:
import json
ergebnis = {"aggregat": [dict(r, ci=list(r["ci"])) for r in rows]}
if train_set is not None:
    rs = seen["tp"] / (seen["tp"] + seen["fn"]) if seen["tp"] + seen["fn"] else float("nan")
    rn = new["tp"] / (new["tp"] + new["fn"]) if new["tp"] + new["fn"] else float("nan")
    print(f"Recall auf Trainingsformen: {rs:.3f} (n={seen['tp']+seen['fn']})")
    print(f"Recall auf neuen Formen   : {rn:.3f} (n={new['tp']+new['fn']})")
    print(f"False Positives: {fp_seen} Trainingsform / {fp_new} neue Form")
    ergebnis["seen_unseen"] = dict(recall_gesehen=rs, recall_neu=rn,
                                   fp_trainingsform=fp_seen,
                                   fp_neue_form=fp_new)
(AUSGABE / "gold_eval.json").write_text(
    json.dumps(ergebnis, ensure_ascii=False, indent=2), encoding="utf-8")
with open(AUSGABE / "fehlerliste.csv", "w", newline="",
          encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["typ", "erzaehlung", "form", "satz"])
    w.writeheader(); w.writerows(fehler)
print(f"\nGespeichert: {AUSGABE/'gold_eval.json'}, "
      f"{AUSGABE/'fehlerliste.csv'}")

Recall auf Trainingsformen: 0.952 (n=105)
Recall auf neuen Formen   : 0.649 (n=97)
False Positives: 9 Trainingsform / 29 neue Form

Gespeichert: ../05_ergebnisse/ner/gold_eval.json, ../05_ergebnisse/ner/fehlerliste.csv
